# QuickTalk: HellaSwag for run 8 (v2)

Scores run 8's **base model** (after pretraining) and **chat model** on HellaSwag validation (10,042 items, 4 endings,
random = 25%), same scorer as the earlier runs. Results go to `MyDrive/hellaswag_results/` next to run 7 (29.8%),
run 7c (29.7%) and SmolLM2-135M (43.1%), and the last cell prints the full comparison.

Runtime: A100 (or any GPU). About 5-8 min per model on an A100. Only reads the finished checkpoints.

In [ ]:
# 1. Drive, code, libraries
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (will be slow)')
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'; WORK = f'{BASE}/quicktalk_run8'; OUT = f'{BASE}/hellaswag_results'; SRC = '/content/qt_eval'
os.makedirs(OUT, exist_ok=True)
for f in ('checkpoints/pretrain_final.pt', 'checkpoints/sft_final.pt', 'tokenizer.json'):
    assert os.path.exists(f'{WORK}/{f}'), f'missing {WORK}/{f}'
if not os.path.exists(SRC):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'claude/intelligent-ride-oxqjfy', '--filter=blob:none',
                    '--sparse', 'https://github.com/sraivante/quicktalk.git', SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/colab'], check=True)
HS = f'{SRC}/train/colab/hellaswag_eval.py'
!pip -q install datasets tokenizers

In [ ]:
# 2. Run 8 base model (after pretraining): the number to compare with published small models
if not os.path.exists(f'{OUT}/quicktalk_run8_base.json'):
    !python {HS} --workdir {WORK} --ckpt {WORK}/checkpoints/pretrain_final.pt --out {OUT}/quicktalk_run8_base.json

In [ ]:
# 3. Run 8 chat model (same kind of model as the run 7 / 7c numbers)
if not os.path.exists(f'{OUT}/quicktalk_run8_chat.json'):
    !python {HS} --workdir {WORK} --ckpt {WORK}/checkpoints/sft_final.pt --out {OUT}/quicktalk_run8_chat.json

In [ ]:
# 4. Comparison (all results in MyDrive/hellaswag_results; random guessing = 25%)
import json, glob
for p in sorted(glob.glob(f'{OUT}/*.json')):
    r = json.load(open(p)); print(f"{p.split('/')[-1]:34} acc_norm {r['acc_norm']:.1%}  acc {r['acc']:.1%}  ({r['items']} items)")